# Data Preprocessing and the Leakage Rule
## Machine Learning — BS Computer Science 3(3-0)



| | |
|---|---|
| **Notebook** | `NB04_Preprocessing.ipynb` |
| **Lecture** | Lecture 6 (Week 2) |
| **CLO** | CLO-2 |
| **Dataset** | Titanic-style tabular data with missing values and categories |

**Objective.** Build a leakage-proof preprocessing pipeline, and measure what leakage costs you.

**By the end of this notebook you will be able to:**

1. Detect missing values including those disguised as -1, 0 or 9999.
2. Encode ordinal, nominal and class-label data correctly.
3. Choose between normalisation and standardisation.
4. Build a Pipeline and demonstrate experimentally why it prevents leakage.

---

## 0. Setup and reproducibility

In [14]:
# ---------------------------------------------------------------------
# Reproducibility header — required in EVERY notebook you submit
# ---------------------------------------------------------------------
import sys, numpy as np, pandas as pd, sklearn, matplotlib
import matplotlib.pyplot as plt

print('python      ', sys.version.split()[0])
print('numpy       ', np.__version__)
print('pandas      ', pd.__version__)
print('scikit-learn', sklearn.__version__)
print('matplotlib  ', matplotlib.__version__)

RANDOM_STATE = 1
rng = np.random.default_rng(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

python       3.13.7
numpy        2.3.5
pandas       2.3.3
scikit-learn 1.7.2
matplotlib   3.10.7


In [15]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                   OneHotEncoder, LabelEncoder)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

## 1. A deliberately messy dataset

Real data has blanks, categories, mixed scales — and missing values that `isnull()` cannot see.

In [17]:
n = 800
raw = pd.DataFrame({
    'age':      rng.normal(38, 13, n).round(0),
    'fare':     rng.lognormal(3.0, 0.9, n).round(2),
    'sibsp':    rng.integers(0, 5, n),
    'pclass':   rng.choice([1, 2, 3], n, p=[0.2, 0.3, 0.5]),
    'sex':      rng.choice(['male', 'female'], n, p=[0.62, 0.38]),
    'embarked': rng.choice(['S', 'C', 'Q'], n, p=[0.7, 0.2, 0.1]),
    'deck_size': rng.choice(['S', 'M', 'L', 'XL'], n),
})
logit = (-1.2 + 1.8 * (raw.sex == 'female') - 0.5 * (raw.pclass - 1)
         + 0.012 * (40 - raw.age) + 0.01 * raw.fare)
raw['survived'] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

# inject realistic missingness
raw.loc[rng.choice(n, 120, replace=False), 'age'] = np.nan          # visible NaN
raw.loc[rng.choice(n, 60,  replace=False), 'fare'] = -1             # DISGUISED
raw.loc[rng.choice(n, 25,  replace=False), 'embarked'] = np.nan

display(raw.head())
print(raw.shape)

,age,fare,sibsp,pclass,sex,embarked,deck_size,survived
0,30.0,18.77,2,3,female,S,XL,0
1,NaN,15.41,0,3,female,S,L,1
2,NaN,8.82,1,3,female,S,XL,1
3,55.0,28.31,2,2,female,S,M,0
4,NaN,21.80,0,3,female,S,L,0


(800, 8)


## 2. Finding the missing values — including the hidden ones

Look at **proportions**, not counts. Then hunt for disguised codes: `-1` for an unknown age, `0` for an unrecorded measurement, `9999` for unknown income.

In [18]:
print('visible missingness (proportion):')
print(raw.isnull().mean().round(3))
print()
print('describe() — look for impossible minima:')
display(raw.describe().T[['min', 'max', 'mean']].round(2))

visible missingness (proportion):
age          0.150
fare         0.000
sibsp        0.000
pclass       0.000
sex          0.000
embarked     0.031
deck_size    0.000
survived     0.000
dtype: float64

describe() — look for impossible minima:


,min,max,mean
age,-1.0,77.00,38.46
fare,-1.0,257.44,26.58
sibsp,0.0,4.00,2.01
pclass,1.0,3.00,2.30
survived,0.0,1.00,0.35


> **`fare` has a minimum of −1.** A fare cannot be negative: this is a disguised missing value. Left untreated, the model would learn that very low fares predict the outcome.



### Exercise 2.1 — convert it

In [19]:
import numpy as np
# TODO:replace the -1 values in 'fare' with np.nan
raw['fare'] = raw['fare'].replace(-1, np.nan)
# TODO : re-print raw.isnull().mean() and confirm 'fare' now shows missingness
print(raw.isnull().mean().round(3))




age          0.150
fare         0.075
sibsp        0.000
pclass       0.000
sex          0.000
embarked     0.031
deck_size    0.000
survived     0.000
dtype: float64


## 3. Encoding the three kinds of categorical data

**Ordinal** → ordered integers (only you know the order). **Class labels** → `LabelEncoder`. **Nominal** → one-hot, because integer-encoding red/green/blue tells the model that blue > green > red.

In [20]:
size_map = {'S': 0, 'M': 1, 'L': 2, 'XL': 3}       # ORDINAL: order is real
raw['deck_size_ord'] = raw['deck_size'].map(size_map)

print('ordinal mapping check:')
display(raw[['deck_size', 'deck_size_ord']].drop_duplicates()
        .sort_values('deck_size_ord'))

print('\none-hot on a NOMINAL feature:')
display(pd.get_dummies(raw[['embarked']], drop_first=True).head())

ordinal mapping check:


,deck_size,deck_size_ord
5,S,0
3,M,1
1,L,2
0,XL,3



one-hot on a NOMINAL feature:


,embarked_Q,embarked_S
0,False,True
1,False,True
2,False,True
3,False,True
4,False,True


**Why `drop_first=True`?** With three categories the three binary columns always sum to 1, so any one is exactly determined by the other two. Dropping one removes that redundancy — it matters for linear models and for interpreting coefficients.

## 4. The pipeline — making leakage structurally impossible

**Split first. Then preprocess inside a Pipeline.** Any statistic computed from the data — a mean, a standard deviation, a category list — must come from training data only.

In [7]:
raw['fare'] = raw['fare'].replace(-1, np.nan)      # in case exercise 2.1 was skipped

num_cols = ['age', 'fare', 'sibsp', 'deck_size_ord']
cat_cols = ['sex', 'embarked', 'pclass']

X = raw[num_cols + cat_cols]
y = raw['survived'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

numeric = Pipeline([('impute', SimpleImputer(strategy='median',
                                             add_indicator=True)),
                    ('scale',  StandardScaler())])

categorical = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                        ('onehot', OneHotEncoder(handle_unknown='ignore',
                                                 drop='first'))])

pre = ColumnTransformer([('num', numeric,     num_cols),
                         ('cat', categorical, cat_cols)])

pipe = Pipeline([('pre', pre),
                 ('clf', LogisticRegression(max_iter=1000,
                                            random_state=RANDOM_STATE))])

pipe.fit(X_train, y_train)
print('test accuracy %.3f' % pipe.score(X_test, y_test))

scores = cross_val_score(pipe, X_train, y_train, cv=5)
print('CV accuracy   %.3f +/- %.3f' % (scores.mean(), scores.std()))

test accuracy 0.705
CV accuracy   0.720 +/- 0.025


> `add_indicator=True` keeps a binary column recording that the value **was** missing. The fact of absence is often itself predictive, and this costs nothing.

## 5. Experiment — measure what leakage buys you

Run the same model two ways across 20 seeds: scaling the whole dataset **before** the split, versus scaling inside the pipeline **after** it.

In [8]:
from sklearn.preprocessing import StandardScaler as SS

X_num = raw[num_cols].fillna(raw[num_cols].median()).values
y_all = raw['survived'].values

wrong, right = [], []
for seed in range(20):
    # WRONG — scale everything, then split
    Xs = SS().fit_transform(X_num)
    a, b, ya, yb = train_test_split(Xs, y_all, test_size=0.25,
                                    random_state=seed, stratify=y_all)
    m = LogisticRegression(max_iter=1000).fit(a, ya)
    wrong.append(m.score(b, yb))

    # RIGHT — split, then scale inside a pipeline
    a, b, ya, yb = train_test_split(X_num, y_all, test_size=0.25,
                                    random_state=seed, stratify=y_all)
    p = Pipeline([('sc', SS()), ('clf', LogisticRegression(max_iter=1000))])
    p.fit(a, ya)
    right.append(p.score(b, yb))

print('WRONG (scale before split): %.4f +/- %.4f' % (np.mean(wrong), np.std(wrong)))
print('RIGHT (scale in pipeline) : %.4f +/- %.4f' % (np.mean(right), np.std(right)))
print('difference                : %+.4f' % (np.mean(wrong) - np.mean(right)))

WRONG (scale before split): 0.6598 +/- 0.0094
RIGHT (scale in pipeline) : 0.6598 +/- 0.0094
difference                : +0.0000


### Interpretation

The gap from **scaling alone** is usually small — and that is exactly what makes it dangerous: there is no error message, no warning, just a slightly flattering number.

*Write one paragraph on the direction of the difference and why it appears. Then predict what would happen if the leaking step were **feature selection** rather than scaling — you will verify that prediction in NB16.*

The results show the difference between scaling the complete dataset before splitting and scaling the data inside a pipeline after splitting. When scaling is performed before splitting, the scaler uses information from both training and testing data to calculate the mean and standard deviation. This causes data leakage because information from the test set influences preprocessing. When scaling is performed inside a pipeline, the scaler learns only from the training data and then transforms the test data using the same learned values. The difference in accuracy may be small because scaling alone usually has a limited effect on logistic regression. However, the correct approach is to split the data first and perform preprocessing inside a pipeline. If feature selection were performed before splitting, leakage could have a much larger effect because the selection process might use the target labels of the entire dataset, including the test set, making the model's performance appear better than it really is.


---
## Deliverable

Exercise 2.1, the complete ColumnTransformer pipeline with its cross-validated score reported as mean ± std, and the 20-seed leakage experiment with a written interpretation.

---
## Submission checklist

Tick every box before you submit. Items 1–6 are the course reproducibility
rules from Lecture 3 and are marked in the rubric.

- [ ] The random seed is fixed **and printed**
- [ ] Library versions are printed in the first cell
- [ ] The raw data file was **not** modified
- [ ] Preprocessing happens **after** the split, inside a `Pipeline`
- [ ] The dataset source and licence are stated in a markdown cell
- [ ] **Kernel → Restart & Run All** completes without error
- [ ] Every experiment is followed by a written interpretation
- [ ] At least one wrong prediction or failure case is explained
- [ ] Results are reported as **mean ± standard deviation** where cross-validated
- [ ] Every plot has axis labels with units and a legend
